# ⚖️ Day 02a: CAP Theorem & Consistency Models

---

## 🎯 What You'll Master Today
- CAP theorem — the fundamental trade-off
- Consistency models: strong, eventual, causal
- Real-world examples of CAP choices
- PACELC theorem extension

**Goal:** Explain CAP trade-offs fluently in design interviews.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  CAP THEOREM                                                     ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  In a distributed system, you can only guarantee                ║
║  TWO out of THREE:                                               ║
║                                                                   ║
║         C — Consistency                                          ║
║            Every read returns the latest write                  ║
║                                                                   ║
║         A — Availability                                         ║
║            Every request gets a response (even if stale)        ║
║                                                                   ║
║         P — Partition Tolerance                                  ║
║            System works despite network failures                ║
║                                                                   ║
║              C                                                    ║
║             / \                                                   ║
║            /   \                                                  ║
║           / CP  \                                                 ║
║          /       \                                                ║
║         P ─────── A                                              ║
║              AP                                                   ║
║                                                                   ║
║  Since P is unavoidable (networks WILL fail):                   ║
║    Real choice = CP (consistency) or AP (availability)          ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# 1. CAP — Real-World Examples
# ============================================================

cap_examples = {
    "CP Systems (Consistency + Partition Tolerance)": {
        "description": "During partition, reject requests rather than serve stale data",
        "examples": [
            "MongoDB (in default config) — blocks writes during partition",
            "HBase — strong consistency, may be unavailable",
            "Redis (cluster mode) — rejects writes when quorum lost",
            "Zookeeper — consensus-based, stops if no majority",
        ],
        "use_when": "Data correctness > uptime (banking, inventory)",
    },
    "AP Systems (Availability + Partition Tolerance)": {
        "description": "During partition, serve possibly stale data rather than error",
        "examples": [
            "Cassandra — always writable, eventual consistency",
            "DynamoDB — highly available, configurable consistency",
            "CouchDB — eventual consistency, multi-master",
            "DNS — always responds, may be stale (TTL)",
        ],
        "use_when": "Uptime > perfect consistency (social media, CDN)",
    },
    "CA Systems (Consistency + Availability)": {
        "description": "Only possible WITHOUT network partitions (single node)",
        "examples": [
            "Single-node PostgreSQL",
            "Single-node MySQL",
        ],
        "use_when": "Not really applicable in distributed systems",
    },
}

for category, info in cap_examples.items():
    print(f"\n{'='*60}")
    print(f"  {category}")
    print(f"{'='*60}")
    print(f"  {info['description']}")
    print(f"  Examples:")
    for ex in info['examples']:
        print(f"    • {ex}")
    print(f"  Use when: {info['use_when']}")

In [ ]:
# ============================================================
# 2. Consistency Models
# ============================================================

print("""
CONSISTENCY MODELS (from strongest to weakest)
═══════════════════════════════════════════════

1. STRONG CONSISTENCY (Linearizability)
   • After write completes, ALL reads see the new value
   • Like a single-server database
   • Example: Bank balance must be accurate instantly
   • Trade-off: Higher latency, lower availability

2. SEQUENTIAL CONSISTENCY
   • All nodes see operations in SAME order
   • But may not be real-time order
   • Example: Social media feed — order matters, slight delay OK

3. CAUSAL CONSISTENCY
   • Causally related operations seen in order
   • Unrelated operations can be in any order
   • Example: Reply appears after original comment
   • Good balance of consistency and performance

4. READ-YOUR-WRITES CONSISTENCY
   • After YOU write, YOU always see your own write
   • Others may see old value temporarily
   • Example: After updating profile, YOU see changes instantly

5. EVENTUAL CONSISTENCY
   • Given enough time with no new writes, all replicas converge
   • No timing guarantee
   • Example: DNS propagation, social media likes count
   • Highest availability, lowest latency

  Strong ◄────────────────────────────► Eventual
  (slow, safe)              (fast, might be stale)
""")

In [ ]:
# ============================================================
# 3. PACELC Theorem (CAP Extension)
# ============================================================

print("""
╔═══════════════════════════════════════════════════════════════╗
║  PACELC — The Complete Picture                               ║
╠═══════════════════════════════════════════════════════════════╣
║                                                               ║
║  CAP only discusses behavior DURING partitions                ║
║  PACELC adds: what about when there's NO partition?          ║
║                                                               ║
║  if Partition:                                                ║
║      choose Availability or Consistency  (same as CAP)       ║
║  else (normal operation):                                    ║
║      choose Latency or Consistency                           ║
║                                                               ║
║  Examples:                                                    ║
║  ┌──────────┬─────────────┬───────────────┐                  ║
║  │ System   │ Partition   │ Normal (E)    │                  ║
║  ├──────────┼─────────────┼───────────────┤                  ║
║  │ DynamoDB │ PA (avail.) │ EL (latency)  │  PA/EL          ║
║  │ Cass-    │ PA (avail.) │ EL (latency)  │  PA/EL          ║
║  │ andra    │             │               │                  ║
║  │ MongoDB  │ PC (consist)│ EC (consist)  │  PC/EC          ║
║  │ MySQL    │ PC (consist)│ EC (consist)  │  PC/EC          ║
║  │ Cosmos   │ PA or PC    │ EL or EC      │  Configurable!  ║
║  └──────────┴─────────────┴───────────────┘                  ║
║                                                               ║
║  💡 Most modern systems are configurable — you choose the    ║
║     consistency level per operation                          ║
║                                                               ║
╚═══════════════════════════════════════════════════════════════╝
""")

In [ ]:
# ============================================================
# 4. Choosing Consistency — Interview Decision Framework
# ============================================================

scenarios = [
    ("Bank transfer", "Strong", "Money must be accurate. User sees exact balance after transfer."),
    ("Stock trading", "Strong", "Portfolio value must reflect trades immediately."),
    ("Inventory count", "Strong", "Can't oversell. Must know exact count."),
    ("Social media likes", "Eventual", "1,234 vs 1,235 likes — nobody notices a brief delay."),
    ("DNS lookup", "Eventual", "Stale for minutes is fine. TTL-based."),
    ("Search index", "Eventual", "Slight delay before new content appears in search."),
    ("User profile update", "Read-your-writes", "YOU see your changes. Others see them eventually."),
    ("Comment replies", "Causal", "Reply must appear after parent. Other comments can be in any order."),
    ("Shopping cart", "Eventual + merge", "Cart syncs across devices. Conflicts resolved by union."),
    ("Leaderboard", "Eventual", "Slight staleness is OK. Recalculate periodically."),
]

print(f"{'Scenario':<25} {'Consistency':<20} {'Reasoning'}")
print(f"{'─'*25} {'─'*20} {'─'*45}")
for scenario, consistency, reason in scenarios:
    print(f"{scenario:<25} {consistency:<20} {reason}")

print("\n💡 Interview tip: Always explain WHY you chose a consistency level.")
print("   'For X feature, we can tolerate Y seconds of staleness because Z.'")

---

## 🗺️ CAP Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  CAP & CONSISTENCY ESSENTIALS                                    ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  CAP: Pick 2 of 3 (but P is mandatory → choose C or A)         ║
║    CP = consistent but may be unavailable during partition      ║
║    AP = available but may serve stale data during partition     ║
║                                                                   ║
║  Consistency spectrum:                                           ║
║    Strong → Sequential → Causal → Read-your-writes → Eventual ║
║                                                                   ║
║  PACELC: Partition? A/C. Else? L/C.                             ║
║  Money/inventory = strong. Social/search = eventual.            ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Explain CAP theorem | Can't have all 3 in distributed system. P is must → real choice is CP or AP |
| 2 | CP vs AP — when to choose? | Money/inventory → CP. Social media/search → AP |
| 3 | What is eventual consistency? | All replicas converge given time. No timing guarantee. Most scalable |
| 4 | What is PACELC? | Extends CAP: during Partition choose A/C, Else choose L/C. Adds normal-operation trade-off |
| 5 | Pick consistency for user profile updates? | Read-your-writes: user sees own changes instantly, others see eventually. Best UX trade-off |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • CAP: P is mandatory → choose C or A                  │
## │  • CP: consistent, may reject requests (banking)        │
## │  • AP: available, may serve stale data (social media)   │
## │  • Consistency spectrum: strong → eventual               │
## │  • Always explain WHY you chose a consistency level     │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Latency & Throughput** — numbers every programmer should know